In [1]:
import json
import logging
import mimetypes
import os
from argparse import Namespace
from http import HTTPStatus
from pathlib import Path
from typing import Any, Callable, Dict, List
from urllib.parse import urlencode
from wsgiref.simple_server import make_server

from sqllineage import DEFAULT_DIALECT, DEFAULT_HOST, DEFAULT_PORT, STATIC_FOLDER
from sqllineage.config import SQLLineageConfig
from sqllineage.core.metadata.dummy import DummyMetaDataProvider
from sqllineage.exceptions import SQLLineageException
from sqllineage.utils.constant import LineageLevel
from sqllineage.utils.helpers import extract_sql_from_args

logger = logging.getLogger(__name__)


class SQLLineageApp:
    """ 
    SQLLineageApp: A simple flask-like wsgi application to serve static files and handle lineage requests.
    """
    def __init__(self) -> None:
        # save route path 
        self.routes: Dict[str, Callable[[Dict[str, Any]], Dict[str, Any]]] = {}
        self.root_path = Path(SQLLineageConfig.DIRECTORY)
        self.metadata_provider = DummyMetaDataProvider()

    def route(self, path: str):
        def wrapper(handler):
            self.routes[path] = handler
            return handler

        return wrapper

    def __call__(self, environ, start_response) -> List[bytes]:
        static_folder = Path(os.path.dirname(__file__)).joinpath(Path(STATIC_FOLDER))
        request_method = environ["REQUEST_METHOD"]
        path_info = environ["PATH_INFO"]
        try:
            if request_method == "GET":
                mimetype = "text/html; charset=utf-8"
                if path_info == "/":
                    static_fname = str(static_folder.joinpath(Path("index.html")))
                else:
                    if ".." in path_info:
                        # Do not allow going back to parent path of static folder
                        return self.handle_404(start_response)
                    static_file = static_folder.joinpath(Path(path_info.strip("/")))
                    if static_file.exists():
                        static_fname = str(static_file)
                        optional_mimetype = mimetypes.guess_type(path_info)[0]
                        mimetype = (
                            optional_mimetype
                            if optional_mimetype is not None
                            else mimetype
                        )
                    else:
                        return self.handle_404(start_response)
                with open(static_fname, "rb") as f:
                    text = f.read()
                return self.handle_200_text(start_response, mimetype, text)
            elif request_method == "POST":
                print("routes:", self.routes)
                if path_info in self.routes:
                    request_body_size = int(environ["CONTENT_LENGTH"])
                    request_body = environ["wsgi.input"].read(request_body_size)
                    payload = json.loads(request_body)
                    for param in ["d", "f"]:
                        if param in payload and not str(
                            Path(payload[param]).absolute()
                        ).startswith(str(Path(self.root_path).absolute())):
                            return self.handle_403(start_response)
                    data = self.routes[path_info](payload)
                    # print("data:", data)
                    return self.handle_200_json(start_response, data)
                else:
                    return self.handle_404(start_response)
            elif request_method == "OPTIONS":
                if path_info in self.routes:
                    start_response(
                        "200 OK",
                        [
                            ("Access-Control-Allow-Origin", "*"),
                            (
                                "Access-Control-Allow-Headers",
                                "Content-Type",
                            ),
                            ("Access-Control-Allow-Methods", "POST"),
                        ],
                    )
                    return []
                else:
                    return self.handle_404(start_response)
            else:
                return self.handle_405(start_response)
        except (SystemExit, IsADirectoryError, FileNotFoundError, PermissionError):
            return self.handle_404(start_response)
        except (SQLLineageException, RuntimeError) as e:
            return self.handle_400(start_response, str(e))

    @staticmethod
    def handle_200_text(start_response, mimetype, text) -> List[bytes]:
        status_code = HTTPStatus.OK
        start_response(
            f"{status_code.value} {status_code.phrase}", [("Content-type", mimetype)]
        )
        return [text]

    def handle_200_json(self, start_response, data) -> List[bytes]:
        return self.handle_json_response(start_response, HTTPStatus.OK, data)

    def handle_400(self, start_response, message) -> List[bytes]:
        return self.handle_client_error_response(
            start_response, HTTPStatus.BAD_REQUEST, message
        )

    def handle_403(self, start_response) -> List[bytes]:
        message = "File Not Allowed For Accessing"
        return self.handle_client_error_response(
            start_response, HTTPStatus.FORBIDDEN, message
        )

    def handle_404(self, start_response) -> List[bytes]:
        message = "File Not Found"
        return self.handle_client_error_response(
            start_response, HTTPStatus.NOT_FOUND, message
        )

    def handle_405(self, start_response) -> List[bytes]:
        message = "Method Not Allowed"
        return self.handle_client_error_response(
            start_response, HTTPStatus.METHOD_NOT_ALLOWED, message
        )

    def handle_client_error_response(
        self, start_response, status_code, message
    ) -> List[bytes]:
        data = {"message": message}
        return self.handle_json_response(start_response, status_code, data)

    @staticmethod
    def handle_json_response(start_response, status_code, data) -> List[bytes]:
        start_response(
            f"{status_code.value} {status_code.phrase}",
            [
                ("Content-type", "application/json"),
                ("Access-Control-Allow-Origin", "*"),
            ],
        )
        return [json.dumps(data).encode("utf-8")]


app = SQLLineageApp()


@app.route("/lineage")
def lineage(payload):
    # this is to avoid circular import
    from sqllineage.runner import LineageRunner

    req_args = Namespace(**payload)
    sql = extract_sql_from_args(req_args)
    dialect = getattr(req_args, "dialect", DEFAULT_DIALECT)
    lr = LineageRunner(
        sql, dialect=dialect, verbose=True, metadata_provider=app.metadata_provider
    )
    data = {
        "verbose": str(lr),
        "dag": lr.to_cytoscape(),
        "column": lr.to_cytoscape(LineageLevel.COLUMN),
    }
    return data


@app.route("/script")
def script(payload):
    req_args = Namespace(**payload)
    sql = extract_sql_from_args(req_args)
    return {"content": sql}


@app.route("/directory")
def directory(payload):
    if payload.get("f"):
        root = Path(payload["f"]).parent
    elif payload.get("d"):
        root = Path(payload["d"])
    else:
        root = Path(SQLLineageConfig.DIRECTORY)
    data = {
        "id": str(root),
        "name": root.name,
        "is_dir": True,
        "children": [
            {"id": str(p), "name": p.name, "is_dir": p.is_dir()}
            for p in sorted(root.iterdir(), key=lambda _: (not _.is_dir(), _.name))
        ],
    }
    return data


def draw_lineage_graph(**kwargs) -> None:
    host = kwargs.pop("host", DEFAULT_HOST) 
    port = kwargs.pop("port", DEFAULT_PORT)
    querystring = urlencode({k: v for k, v in kwargs.items() if v}) # 将字典转换为url参数
    path = f"/?{querystring}" if querystring else "/" # 生成url
    if f := kwargs.get("f"): # 获取文件路径
        app.root_path = Path(f).parent # 设置文件路径
    if metadata_provider := kwargs.get("metadata_provider"): # 获取元数据
        app.metadata_provider = metadata_provider # 设置元数据
    with make_server(host, port, app) as httpd: # 启动服务 
        print(f" * SQLLineage Running on http://{host}:{port}{path}") # 打印服务地址
        httpd.serve_forever()  # 服务一直运行


In [2]:
## read sql file to string
sql_path = '/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data/vgds/app_rt_event_fod_ot.sql'
with open(sql_path, 'r') as f:
    sql = f.read()

In [3]:
from sqllineage.runner import LineageRunner
result = LineageRunner(sql,dialect='non-validating')
print(result)

Statements(#): 2
Source Tables:
    vgds.app_trip_rt_event_fod_stats
Target Tables:
    vgds.app_trip_rt_event_fod_ot



/tmp/ipykernel_1418978/2817020059.py:2: DeprecationWarning: dialect `non-validating` is deprecated, use `ansi` or dialect of your SQL instead. `non-validating` will be completely removed in v1.6.x
  result = LineageRunner(sql,dialect='non-validating')


In [4]:
result.draw()


 * SQLLineage Running on http://localhost:5001/?e=--SPARK_SQL%0A--%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A--%0A--author%3Ayanranhan%0A--create+time%3A2024-07-12+02%3A21%3A31%0A--desc%3A%E5%87%BA%E5%A2%83%E8%84%B1%E6%95%8F%E8%A1%A8%0A--remind%3A%E8%AF%B7%E5%9C%A8%E8%B5%84%E6%BA%90%E5%BC%95%E7%94%A8%E4%B8%AD%E6%B7%BB%E5%8A%A0%E9%9C%80%E8%A6%81%E5%BC%95%E7%94%A8%E7%9A%84%E8%B5%84%E6%BA%90%0A--%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A%2A--%0Acreate+table+if+not+exists+app_trip_rt_event_fod_ot%0A%28%0A+trip_id+string%0A+%2Ctrip_category_name+string%0A+%2Cregion+string%0A+%2Cbig_version+string%0A+%2Ctest_version+string%0A+%2Cpeak+string%0A+%2Cactual_engage_distance+float%0A

127.0.0.1 - - [12/Oct/2024 17:25:43] "GET /?f=/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data/vgds/app_ds_asm_fallback_hb_hs_collision_missframe.sql HTTP/1.1" 200 736
127.0.0.1 - - [12/Oct/2024 17:25:43] "GET /static/js/main.e032b9fa.js HTTP/1.1" 200 3257206
127.0.0.1 - - [12/Oct/2024 17:25:43] "GET /static/css/main.84d1d546.css HTTP/1.1" 200 65835
127.0.0.1 - - [12/Oct/2024 17:25:43] "GET /static/js/333.140e3456.chunk.js HTTP/1.1" 200 18267
127.0.0.1 - - [12/Oct/2024 17:25:43] "POST /directory HTTP/1.1" 200 79130
127.0.0.1 - - [12/Oct/2024 17:25:43] "POST /script HTTP/1.1" 200 3118
127.0.0.1 - - [12/Oct/2024 17:25:43] "POST /lineage HTTP/1.1" 200 19692
127.0.0.1 - - [12/Oct/2024 17:25:43] "POST /script HTTP/1.1" 200 3118


routes: {'/lineage': <function lineage at 0x77c899bfae80>, '/lineageall': <function lineage at 0x77c899bfaf20>, '/script': <function script at 0x77c899bfafc0>, '/scriptall': <function scriptall at 0x77c899bfb060>, '/directory': <function directory at 0x77c899bfb100>}
routes: {'/lineage': <function lineage at 0x77c899bfae80>, '/lineageall': <function lineage at 0x77c899bfaf20>, '/script': <function script at 0x77c899bfafc0>, '/scriptall': <function scriptall at 0x77c899bfb060>, '/directory': <function directory at 0x77c899bfb100>}
routes: {'/lineage': <function lineage at 0x77c899bfae80>, '/lineageall': <function lineage at 0x77c899bfaf20>, '/script': <function script at 0x77c899bfafc0>, '/scriptall': <function scriptall at 0x77c899bfb060>, '/directory': <function directory at 0x77c899bfb100>}
Statements(#): 1
Source Tables:
    
Target Tables:
    vgds.app_ds_asm_fallback_hb_hs_collision_missframe

Statements(#): 1
Source Tables:
    vgds.dwd_ds_asm_fallback_collision_tp_di
    vgds.dw

127.0.0.1 - - [12/Oct/2024 17:25:43] "POST /lineage HTTP/1.1" 200 19692
127.0.0.1 - - [12/Oct/2024 17:25:43] "GET /editor.worker.js HTTP/1.1" 200 121291
127.0.0.1 - - [12/Oct/2024 17:25:44] "GET /favicon.ico HTTP/1.1" 200 3870
127.0.0.1 - - [12/Oct/2024 17:25:44] "GET /manifest.json HTTP/1.1" 200 494
127.0.0.1 - - [12/Oct/2024 17:25:44] "GET /logo192.png HTTP/1.1" 200 5347
127.0.0.1 - - [12/Oct/2024 17:27:50] "GET /?f=/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data/vgds/app_ds_asm_fallback_hb_hs_collision_missframe.sql HTTP/1.1" 200 736
127.0.0.1 - - [12/Oct/2024 17:27:50] "GET /static/js/main.73865042.js HTTP/1.1" 200 3257092
127.0.0.1 - - [12/Oct/2024 17:27:50] "GET /static/css/main.84d1d546.css HTTP/1.1" 200 65835
127.0.0.1 - - [12/Oct/2024 17:27:50] "GET /static/js/333.140e3456.chunk.js HTTP/1.1" 200 18267
127.0.0.1 - - [12/Oct/2024 17:27:50] "POST /directory HTTP/1.1" 200 79130
127.0.0.1 - - [12/Oct/2024 17:27:50] "POST /script HTTP/1.1" 200 3118
127.0.0.1 - - [12

routes: {'/lineage': <function lineage at 0x77c899bfae80>, '/lineageall': <function lineage at 0x77c899bfaf20>, '/script': <function script at 0x77c899bfafc0>, '/scriptall': <function scriptall at 0x77c899bfb060>, '/directory': <function directory at 0x77c899bfb100>}
routes: {'/lineage': <function lineage at 0x77c899bfae80>, '/lineageall': <function lineage at 0x77c899bfaf20>, '/script': <function script at 0x77c899bfafc0>, '/scriptall': <function scriptall at 0x77c899bfb060>, '/directory': <function directory at 0x77c899bfb100>}
routes: {'/lineage': <function lineage at 0x77c899bfae80>, '/lineageall': <function lineage at 0x77c899bfaf20>, '/script': <function script at 0x77c899bfafc0>, '/scriptall': <function scriptall at 0x77c899bfb060>, '/directory': <function directory at 0x77c899bfb100>}
Statements(#): 1
Source Tables:
    
Target Tables:
    vgds.app_ds_asm_fallback_hb_hs_collision_missframe

Statements(#): 1
Source Tables:
    vgds.dwd_ds_asm_fallback_collision_tp_di
    vgds.dw

...

127.0.0.1 - - [12/Oct/2024 17:53:21] "POST /lineage HTTP/1.1" 200 6372
127.0.0.1 - - [12/Oct/2024 17:53:21] "GET /editor.worker.js HTTP/1.1" 200 121291
127.0.0.1 - - [12/Oct/2024 17:53:21] "GET /favicon.ico HTTP/1.1" 200 3870
127.0.0.1 - - [12/Oct/2024 17:53:21] "GET /manifest.json HTTP/1.1" 200 494
127.0.0.1 - - [12/Oct/2024 17:53:22] "GET /logo192.png HTTP/1.1" 200 5347
127.0.0.1 - - [12/Oct/2024 17:54:45] "GET /?f=/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data/vgds/adjusted_play_cnt.sql HTTP/1.1" 200 736
127.0.0.1 - - [12/Oct/2024 17:54:45] "GET /static/js/main.49d0375b.js HTTP/1.1" 200 3257141
127.0.0.1 - - [12/Oct/2024 17:54:45] "GET /static/css/main.84d1d546.css HTTP/1.1" 200 65835
127.0.0.1 - - [12/Oct/2024 17:54:46] "GET /static/js/333.140e3456.chunk.js HTTP/1.1" 200 18267
127.0.0.1 - - [12/Oct/2024 17:54:46] "POST /directory HTTP/1.1" 200 79130
127.0.0.1 - - [12/Oct/2024 17:54:46] "POST /script HTTP/1.1" 200 3276
127.0.0.1 - - [12/Oct/2024 17:54:46] "POST /li

routes: {'/lineage': <function lineage at 0x77c899bfae80>, '/lineageall': <function lineage at 0x77c899bfaf20>, '/script': <function script at 0x77c899bfafc0>, '/scriptall': <function scriptall at 0x77c899bfb060>, '/directory': <function directory at 0x77c899bfb100>}
routes: {'/lineage': <function lineage at 0x77c899bfae80>, '/lineageall': <function lineage at 0x77c899bfaf20>, '/script': <function script at 0x77c899bfafc0>, '/scriptall': <function scriptall at 0x77c899bfb060>, '/directory': <function directory at 0x77c899bfb100>}
routes: {'/lineage': <function lineage at 0x77c899bfae80>, '/lineageall': <function lineage at 0x77c899bfaf20>, '/script': <function script at 0x77c899bfafc0>, '/scriptall': <function scriptall at 0x77c899bfb060>, '/directory': <function directory at 0x77c899bfb100>}
Statements(#): 1
Source Tables:
    
Target Tables:
    vgds.app_ares_adjusted_play_cnt

Statements(#): 1
Source Tables:
    omega.ods_log_web_internal_increment
    vgds.app_ares_adjusted_play_cn

127.0.0.1 - - [12/Oct/2024 17:54:46] "POST /lineage HTTP/1.1" 200 6372
127.0.0.1 - - [12/Oct/2024 17:54:46] "GET /editor.worker.js HTTP/1.1" 200 121291
127.0.0.1 - - [12/Oct/2024 17:54:46] "GET /favicon.ico HTTP/1.1" 200 3870
127.0.0.1 - - [12/Oct/2024 17:54:46] "GET /manifest.json HTTP/1.1" 200 494
127.0.0.1 - - [12/Oct/2024 17:54:47] "GET /logo192.png HTTP/1.1" 200 5347
127.0.0.1 - - [12/Oct/2024 18:04:07] "GET /?f=/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data/vgds/adjusted_play_cnt.sql HTTP/1.1" 200 736
127.0.0.1 - - [12/Oct/2024 18:04:07] "GET /static/js/main.325f9bf0.js HTTP/1.1" 200 3258715
127.0.0.1 - - [12/Oct/2024 18:04:07] "GET /static/css/main.84d1d546.css HTTP/1.1" 200 65835
127.0.0.1 - - [12/Oct/2024 18:04:08] "GET /static/js/333.140e3456.chunk.js HTTP/1.1" 200 18267
127.0.0.1 - - [12/Oct/2024 18:04:08] "POST /directory HTTP/1.1" 200 79130
127.0.0.1 - - [12/Oct/2024 18:04:08] "POST /script HTTP/1.1" 200 3276
127.0.0.1 - - [12/Oct/2024 18:04:08] "POST /li

routes: {'/lineage': <function lineage at 0x77c899bfae80>, '/lineageall': <function lineage at 0x77c899bfaf20>, '/script': <function script at 0x77c899bfafc0>, '/scriptall': <function scriptall at 0x77c899bfb060>, '/directory': <function directory at 0x77c899bfb100>}
routes: {'/lineage': <function lineage at 0x77c899bfae80>, '/lineageall': <function lineage at 0x77c899bfaf20>, '/script': <function script at 0x77c899bfafc0>, '/scriptall': <function scriptall at 0x77c899bfb060>, '/directory': <function directory at 0x77c899bfb100>}
routes: {'/lineage': <function lineage at 0x77c899bfae80>, '/lineageall': <function lineage at 0x77c899bfaf20>, '/script': <function script at 0x77c899bfafc0>, '/scriptall': <function scriptall at 0x77c899bfb060>, '/directory': <function directory at 0x77c899bfb100>}
Statements(#): 1
Source Tables:
    
Target Tables:
    vgds.app_ares_adjusted_play_cnt

Statements(#): 1
Source Tables:
    omega.ods_log_web_internal_increment
    vgds.app_ares_adjusted_play_cn

127.0.0.1 - - [12/Oct/2024 18:04:08] "POST /lineage HTTP/1.1" 200 6372
127.0.0.1 - - [12/Oct/2024 18:04:08] "GET /editor.worker.js HTTP/1.1" 200 121291
127.0.0.1 - - [12/Oct/2024 18:04:08] "GET /favicon.ico HTTP/1.1" 200 3870
127.0.0.1 - - [12/Oct/2024 18:04:08] "GET /manifest.json HTTP/1.1" 200 494
127.0.0.1 - - [12/Oct/2024 18:04:09] "GET /logo192.png HTTP/1.1" 200 5347


In [ ]:
1